# 06 - Baseline 2: TF-IDF + Balanced Logistic Regression

## Overview
Evaluates a simple, transparent classical ML baseline using sublinear TF-IDF (unigrams + bigrams) and regularized Logistic Regression with inverse frequency class weighting (`class_weight='balanced'`).

### Evaluation Datasets:
1. **Held-out Test Split** (484 real conversations from `data/splits/test.jsonl`)
2. **Golden Evaluation Set** (200 hand-verified conversations from `data/golden/golden_set.jsonl`)

In [ ]:
import json
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

vec_path = Path('../models/intent_classifier/tfidf_vectorizer.joblib')
model_path = Path('../models/intent_classifier/tfidf_logreg_model.joblib')
test_path = Path('../data/splits/test.jsonl')
golden_path = Path('../data/golden/golden_set.jsonl')

vectorizer = joblib.load(vec_path)
classifier = joblib.load(model_path)
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Classes: {classifier.classes_}")

## 1. Feature Importance: Top Informative N-Grams per Intent

In [ ]:
feature_names = vectorizer.get_feature_names_out()
for i, cls in enumerate(classifier.classes_):
    top_idx = classifier.coef_[i].argsort()[-5:][::-1]
    top_ngrams = [feature_names[j] for j in top_idx]
    print(f"{cls:30s} -> {', '.join(top_ngrams)}")

## 2. Evaluation on Held-Out Test Split (484 Samples)

In [ ]:
test_data = [json.loads(l) for l in open(test_path, encoding='utf-8') if l.strip()]
X_test = vectorizer.transform([r['first_inquiry'] for r in test_data])
y_test = [r['intent_code'] for r in test_data]

preds_test = classifier.predict(X_test)
acc_test = accuracy_score(y_test, preds_test)
print(f"Held-Out Test Accuracy: {acc_test * 100:.2f}%")
print(classification_report(y_test, preds_test, zero_division=0))

## 3. Evaluation on Hand-Verified Golden Set (200 Samples)

In [ ]:
golden_data = [json.loads(l) for l in open(golden_path, encoding='utf-8') if l.strip()]
X_gold = vectorizer.transform([r['customer_message'] for r in golden_data])
y_gold = [r['gold_intent_code'] for r in golden_data]

preds_gold = classifier.predict(X_gold)
acc_gold = accuracy_score(y_gold, preds_gold)
print(f"Golden Set Accuracy: {acc_gold * 100:.2f}%")
print(classification_report(y_gold, preds_gold, zero_division=0))